# VFCFinder results vs. ground truth

Compares the top-ranked commit from each `ranked_<GHSA>.csv` (row 1 = predicted VFC) with `VFCCommitSHA` in `tests/data/sample/ghsa_annotation_sample.json`.
Advisories with an empty `VFCCommitSHA` have no ground truth, so they are listed but not scored.

In [16]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

# find the repo root from wherever the notebook is opened
REPO_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tests/data/sample").exists()
)
ANNOTATION_PATH = REPO_ROOT / "tests/data/sample/ghsa_annotation_sample.json"
RESULTS_DIR = REPO_ROOT / "output/results/sample"


## Ground truth

In [17]:
annotations = pd.read_json(ANNOTATION_PATH, dtype=str)
annotations["VFCCommitSHA"] = annotations["VFCCommitSHA"].fillna("").str.strip().str.lower()
annotations["has_ground_truth"] = annotations["VFCCommitSHA"] != ""
annotations[["ghsa_id", "fixed_version", "has_ground_truth", "VFCCommitSHA"]]

,ghsa_id,fixed_version,has_ground_truth,VFCCommitSHA
0,GHSA-xph3-r2jf-4vp3,3.1.4,False,
1,GHSA-4w46-w44m-3jq3,4.5.0,True,da905a357d062ab4fea727a21eac231acc2ed92a
2,GHSA-2phv-j68v-wwqx,10.27.0,False,
3,GHSA-9pp3-53p2-ww9v,3.6.2,False,
4,GHSA-99h5-pjcv-gr6v,1.3.26,True,556085067609c508f8c546ceef9003ee8c607d39
5,GHSA-8xwg-wv7v-4vqp,1.8.4,False,
6,GHSA-5xf4-f2fq-f69j,1.22.0,True,039bafd74b7b1a88a53a54f8fa6fa872615e90e7
7,GHSA-4vq7-882g-wcg4,5.23.0,False,
8,GHSA-v273-448j-v4qj,10.25.5,True,f41c1fc02fe901598f3328118b42b13bc6bc9b04
9,GHSA-49x3-8228-3w3m,3.12.0,True,ca617df135e78833f93c8320cb2d2cf8bba809f5


## VFCFinder predictions

The GHSA ID comes from the file name. Rows are sorted by `ranking_prob` (descending) so the first row is the top-ranked commit.

In [18]:
rows = []
ranked = {}  # ghsa_id -> full ranked dataframe, used for the rank lookup below
for csv_path in sorted(RESULTS_DIR.glob("ranked_*.csv")):
    ghsa_id = csv_path.stem.removeprefix("ranked_")
    df = pd.read_csv(csv_path).sort_values("ranking_prob", ascending=False, kind="stable")
    df = df.reset_index(drop=True)
    ranked[ghsa_id] = df
    top = df.iloc[0]
    rows.append(
        {
            "ghsa_id": ghsa_id,
            "predicted_sha": top["sha"].lower(),
            "ranking_prob": top["ranking_prob"],
            "candidate_commits": len(df),
            "predicted_message": top["message"],
        }
    )

predictions = pd.DataFrame(rows)
predictions[["ghsa_id", "predicted_sha", "ranking_prob", "candidate_commits"]]

,ghsa_id,predicted_sha,ranking_prob,candidate_commits
0,GHSA-2phv-j68v-wwqx,5d983ba175a658dae65597dda0f4de04fdc54948,0.088440,15
1,GHSA-2q7r-29rg-6m5h,4d9795cd5b57a36756d37b7f036eae369f69fa66,0.107930,3
2,GHSA-2qgm-m29m-cj2h,6cfae01a0d3727c517afe512fc8fec1d99acf875,0.160546,3
3,GHSA-49x3-8228-3w3m,1d4ffda1f8a74f9e6804cc843404da3b7e1806fd,0.031484,117
4,GHSA-4hjh-wcwx-xvwj,a1b1d3f073a988601583a604f5f9f5d05a3d0b67,0.060061,4
5,GHSA-4vq7-882g-wcg4,01adb034f24727d3bb321bbbb6696a7f4cd91689,0.107442,47
6,GHSA-4w46-w44m-3jq3,da905a357d062ab4fea727a21eac231acc2ed92a,0.130528,64
7,GHSA-5xf4-f2fq-f69j,1b334e687baa75c84440def211cca0636879d14d,0.056844,18
8,GHSA-8vch-m3f4-q8jf,3af978663e437dccc6c1a2a3aff4b74e1574849e,0.135235,62
9,GHSA-8xwg-wv7v-4vqp,fe2dfdc24d876f4233ab0f27e9759c6956ac5235,0.063388,25


## Comparison - we ignore no ground-truth samples in the comparison

In [19]:
def rank_of(ghsa_id, sha):
    """1-based position of sha in the ranked CSV, None if it is not a candidate"""
    if ghsa_id not in ranked or not sha:
        return None
    matches = ranked[ghsa_id].index[ranked[ghsa_id]["sha"].str.lower() == sha]
    return int(matches[0]) + 1 if len(matches) else None


# only advisories with a ground-truth VFC are compared
comparison = (
    annotations[annotations["has_ground_truth"]]
    .merge(predictions, on="ghsa_id", how="left")
    .reset_index(drop=True)
)
comparison["ground_truth_rank"] = [
    rank_of(r.ghsa_id, r.VFCCommitSHA) for r in comparison.itertuples()
]


def status(r):
    if pd.isna(r["predicted_sha"]):
        return "no result CSV"
    return "match" if r["predicted_sha"] == r["VFCCommitSHA"] else "mismatch"


comparison["status"] = comparison.apply(status, axis=1)

comparison[
    ["ghsa_id", "fixed_version", "status", "VFCCommitSHA", "predicted_sha", "ground_truth_rank", "candidate_commits"]
].rename(columns={"VFCCommitSHA": "ground_truth_sha"})

,ghsa_id,fixed_version,status,ground_truth_sha,predicted_sha,ground_truth_rank,candidate_commits
0,GHSA-4w46-w44m-3jq3,4.5.0,match,da905a357d062ab4fea727a21eac231acc2ed92a,da905a357d062ab4fea727a21eac231acc2ed92a,1,64
1,GHSA-99h5-pjcv-gr6v,1.3.26,match,556085067609c508f8c546ceef9003ee8c607d39,556085067609c508f8c546ceef9003ee8c607d39,1,6
2,GHSA-5xf4-f2fq-f69j,1.22.0,mismatch,039bafd74b7b1a88a53a54f8fa6fa872615e90e7,1b334e687baa75c84440def211cca0636879d14d,11,18
3,GHSA-v273-448j-v4qj,10.25.5,match,f41c1fc02fe901598f3328118b42b13bc6bc9b04,f41c1fc02fe901598f3328118b42b13bc6bc9b04,1,6
4,GHSA-49x3-8228-3w3m,3.12.0,mismatch,ca617df135e78833f93c8320cb2d2cf8bba809f5,1d4ffda1f8a74f9e6804cc843404da3b7e1806fd,2,117
5,GHSA-w87r-vg9q-crqm,8.8.5,match,9ef6d3c9962c4ba01e3fb8075855570c192b4681,9ef6d3c9962c4ba01e3fb8075855570c192b4681,1,14
6,GHSA-2qgm-m29m-cj2h,1.23.16,match,6cfae01a0d3727c517afe512fc8fec1d99acf875,6cfae01a0d3727c517afe512fc8fec1d99acf875,1,3
7,GHSA-2q7r-29rg-6m5h,12.5.0,match,4d9795cd5b57a36756d37b7f036eae369f69fa66,4d9795cd5b57a36756d37b7f036eae369f69fa66,1,3
8,GHSA-4hjh-wcwx-xvwj,0.30.2,match,a1b1d3f073a988601583a604f5f9f5d05a3d0b67,a1b1d3f073a988601583a604f5f9f5d05a3d0b67,1,4
9,GHSA-8vch-m3f4-q8jf,1.4.17,mismatch,26935bf76ebc43b4a43d48b173fc853de43bb51e,3af978663e437dccc6c1a2a3aff4b74e1574849e,2,62
